# 🃏 Poker ML Simulator — Colab Training
### 使用流程
1. **掛載 Google Drive**（checkpoint 存在這裡，Colab 重啟不怕丟失）
2. Clone repo & 安裝依賴
3. 開始訓練 / 繼續訓練
4. 訓練中斷後只要重新執行就能從上次繼續

In [ ]:
# ── Step 1: 掛載 Google Drive ────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Checkpoint 會存在這個目錄，Colab 重啟後仍然保留
CKPT_DIR = '/content/drive/MyDrive/poker_checkpoints'
print(f'Checkpoint dir: {CKPT_DIR}')

In [ ]:
# ── Step 2: Clone repo & 安裝依賴 ───────────────────────────
import os

REPO_DIR = '/content/poker-ml-simulator'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/caizongxun/poker-ml-simulator {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull  # 如果已存在就更新

os.chdir(REPO_DIR)
!pip install -q -r requirements.txt
print('Ready!')

In [ ]:
# ── Step 3: 確認 GPU ────────────────────────────────────────
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Step 4: 訓練（斷點後重執行此格就能繼續）────────────────
# checkpoint 存在 Google Drive，資料集快取也存在同一目錄
# 中斷後重新執行會自動從最新 epoch 繼續

!python -m training.train_supervised \
    --epochs 100 \
    --samples 200000 \
    --mc_sims 150 \
    --workers 2 \
    --ckpt_dir {CKPT_DIR} \
    --save_path {CKPT_DIR}/supervised_model.pt

In [ ]:
# ── Step 5: 查看最新 checkpoint 狀態 ────────────────────────
import json, pathlib

latest = pathlib.Path(CKPT_DIR) / 'latest.json'
if latest.exists():
    info = json.loads(latest.read_text())
    print(f"Last epoch  : {info['epoch']}")
    print(f"Val acc     : {info['val_acc']:.3f}")
    print(f"Checkpoint  : {info['path']}")
else:
    print('No checkpoint found yet.')

## 🖥️ 本地繼續訓練
```bash
# 1. 把 Drive 上的 checkpoint 資料夾複製到本地
#    （Google Drive 桌面版 或 手動下載整個資料夾）

# 2. 在本地繼續
python -m training.train_supervised \
    --epochs 100 \
    --samples 200000 \
    --ckpt_dir ./poker_checkpoints \
    --save_path ./poker_checkpoints/supervised_model.pt
```
> 自動讀取 `latest.json` → 從上次的 epoch 繼續，資料集也從 `.npz` 快取讀取不重新生成